# Tokenizing Text and Aligning Labels

*Chapter 4 - Multilingual Named Entity Recognition · Natural Language Processing with Transformers (Tunstall, von Werra and Wolf)*

## Overview

The NER dataset gives **one label for each word**. A transformer model, however, reads **sub-word tokens**, and one word can become several tokens. If we feed the tokens to the model and keep the word labels as they are, the two sequences have different lengths and the labels do not match the tokens.

This notebook solves the problem. We tokenize a German example, look at how the words are split, and then build a rule that gives every token a valid label. At the end, we pack the rule into two functions and encode the full German corpus.

## Objective

- Tokenize a sentence that is already split into words with the XLM-R tokenizer.
- Use `word_ids()` to see which word each token comes from.
- Align word labels with tokens, using the special value `-100` where no label should be used.
- Create reusable functions that encode a whole corpus in batches.

## Workflow

| Step | What happens | Section |
|---|---|---|
| 1 | Tokenize one German example and print the tokens | Section 1 |
| 2 | Read the word ids of the tokens | Section 1 |
| 3 | Build the label ids and a table with readable labels | Section 2 |
| 4 | Convert the logic into `tokenized_and_align_labels` | Section 3 |
| 5 | Encode the German corpus with `encode_panx_dataset` | Section 3 |

## Tools and inputs

- **`transformers`**: the XLM-R tokenizer `xlmr_tokenizer` (from the first notebook).
- **`datasets`**: `Dataset.map` applies a function to many examples at once.
- **`pandas`**: small tables for the token, word id and label rows.
- **Inputs from `a_glance_at_dataset_and_tokenizer.ipynb`:** `tags`, `de_example`, `xlmr_tokenizer` and `panx_ch`.
- **Outputs used by later notebooks:** `index2tag`, `tag2index`, `encode_panx_dataset` and `panx_de_encoded`.

## Notes for the reader

- `import_ipynb` runs the whole first notebook when this notebook starts. The first cell uses `%%capture`, so the messages of this import are hidden.
- The first cell also sets `USE_TF` to `"0"`, so `transformers` does not load TensorFlow.

In [1]:
%%capture
import os
os.environ["USE_TF"] = "0"

import import_ipynb
from a_glance_at_dataset_and_tokenizer import tags, de_example, xlmr_tokenizer, panx_ch

import pandas as pd

## Section 1 - Tokenizing a Pre-split Example

### 1.1 From Words to Tokens

The German example is already a list of 12 words. With `is_split_into_words=True`, the tokenizer knows this and tokenizes each word on its own, so it can remember which word each token came from. The function `convert_ids_to_tokens` then turns the ids back into readable token strings.

The stored table has **25 tokens** for the 12 words. We can see why:

- The special tokens `<s>` and `</s>` are added at the start and at the end.
- Long or rare words are split. For example, "Einwohnern" becomes `▁Einwohner` and `n`, and "Danziger" becomes `▁Dan`, `zi` and `ger`.
- The final "." becomes two tokens: `▁` and `.`.

In [16]:
# Words and integer tags of the first German training example
words, labels = de_example["tokens"], de_example["ner_tags"]
# is_split_into_words=True: the input is already a list of words
tokenized_input = xlmr_tokenizer(de_example["tokens"], is_split_into_words=True)
tokens = xlmr_tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
pd.DataFrame([tokens], index=["Tokens"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>


### 1.2 Word IDs

`word_ids()` returns one value for each token: the index of the word in the original list, or `None` for a special token. In the stored table, the tokens `▁Dan`, `zi` and `ger` all have the word id `4`. This list is the key for the next step, because it tells us where a new word starts.

In [17]:
# word_ids() gives the index of the word for every token (None for special tokens)
word_ids = tokenized_input.word_ids()
pd.DataFrame([tokens, word_ids], index=['Tokens', "Word IDs"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None


## Section 2 - Aligning Labels with Sub-tokens

### 2.1 The Alignment Rule

Every token needs one label id for the loss function. We use this rule:

| Token type | Label id | Reason |
|---|---|---|
| Special token (`<s>`, `</s>`) | `-100` | It is not a word of the sentence |
| First sub-token of a word | The label of the word | It represents the word |
| Other sub-tokens of a word | `-100` | The word is already labeled once |

The value `-100` is special: PyTorch's `CrossEntropyLoss` ignores targets with this value by default (`ignore_index=-100`). The model still makes a prediction for these positions, but they add nothing to the loss. In the same way, the evaluation code of the next notebooks only uses positions where the label is not `-100`.

### 2.2 Tag Look-up Tables and the Result

The cell first creates two dictionaries, `index2tag` and `tag2index`, from `tags.names`. Other notebooks import `index2tag` to turn ids back into names. Then a loop goes through the word ids and uses the previous word id to decide if a token is the first sub-token of a word.

The table at the end shows the result. For "Danziger Bucht", the token `▁Dan` has the label id `5` (`B-LOC`), and `zi` and `ger` have `-100`. The readable row uses `IGN` for these ignored positions.

In [18]:
# Look-up tables between tag ids and tag names (other notebooks reuse them)
index2tag = {idx: tag for idx, tag in enumerate(tags.names)}
tag2index = {tag: idx for idx, tag in enumerate(tags.names)}

# -100 is ignored by the loss: special tokens and every sub-token after the first
previous_word_idx = None
label_ids = []

for word_idx in word_ids:
    if word_idx is None or word_idx == previous_word_idx:
        label_ids.append(-100)
    elif word_idx != previous_word_idx:
        label_ids.append(labels[word_idx])
    previous_word_idx = word_idx

# Show "IGN" in the table for the positions that the loss ignores
labels = [index2tag[l] if l != -100 else "IGN" for l in label_ids]
index = ["Tokens", "Word IDs", "Label IDs", "Labels"]

pd.DataFrame([tokens, word_ids, label_ids, labels], index=index)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None
Label IDs,-100,0,0,-100,0,0,5,-100,-100,6,...,5,-100,-100,-100,6,-100,-100,0,-100,-100
Labels,IGN,O,O,IGN,O,O,B-LOC,IGN,IGN,I-LOC,...,B-LOC,IGN,IGN,IGN,I-LOC,IGN,IGN,O,IGN,IGN


## Section 3 - Reusable Functions and Encoding the German Corpus

### 3.1 Aligning Labels for a Whole Batch

The function `tokenized_and_align_labels` uses the same rule as before, but for many sentences at the same time. It does three things:

1. It tokenizes the batch with `truncation=True`, so very long sentences are cut to the maximum length of the model.
2. For each sentence, it reads `word_ids(batch_index=idx)` and builds the list of label ids.
3. It stores the lists in a new key called `labels` and returns the tokenizer output.

Note that the argument `examples` is a dictionary of lists, because `Dataset.map(..., batched=True)` gives the function a batch and not a single example.

In [31]:
def tokenized_and_align_labels(examples):
    """Tokenize pre-split sentences and align word labels with sub-tokens.

    Only the first sub-token of each word keeps the label of the word. Special
    tokens and all other sub-tokens get -100, which the loss function ignores.

    Args:
        examples: Batch of examples (a dictionary of lists) with the columns
            `tokens` (list of words) and `ner_tags` (list of integer tag ids).

    Returns:
        The tokenizer output for the batch (`input_ids` and `attention_mask`)
        with an extra `labels` key. Each entry of `labels` has one id for
        every token of the sentence.
    """
    # Tokenize all sentences of the batch; the words are already split
    tokenized_inputs = xlmr_tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    labels = []
    for idx, label in enumerate(examples["ner_tags"]):
        # Word index of every token in the idx-th sentence
        word_ids = tokenized_inputs.word_ids(batch_index=idx)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens and continuation sub-tokens are ignored by the loss
            if word_idx is None or previous_word_idx == word_idx:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

### 3.2 Encoding a Corpus

`encode_panx_dataset` applies the function above to every split with `map(batched=True)`. It removes the columns `tokens`, `ner_tags` and `langs`, because the model does not need them for training. What remains are `input_ids`, `attention_mask` and `labels`. The function takes a whole `DatasetDict`, so one call prepares the train, validation and test splits. Later notebooks call it for French, Italian and English too.

In [32]:
def encode_panx_dataset(corpus):
    """Tokenize a PAN-X corpus and align its labels.

    Args:
        corpus: A `DatasetDict` with the splits `train`, `validation` and
            `test`. Each split has the columns `tokens`, `ner_tags` and `langs`.

    Returns:
        A `DatasetDict` with the columns `input_ids`, `attention_mask` and
        `labels`. The original text columns are removed.
    """
    return corpus.map(
        tokenized_and_align_labels, 
        batched=True,
        remove_columns=["tokens", "ner_tags", "langs"],
    )

### 3.3 Encoding the German Corpus

The progress bars in the stored output show the sizes of the German splits: **12,580** training, **6,290** validation and **6,290** test examples. The result `panx_de_encoded` is the data that the model sees in the fine-tuning and error analysis notebooks.

In [33]:
panx_de_encoded = encode_panx_dataset(panx_ch["de"])

Map:   0%|          | 0/12580 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

## Conclusion

This notebook connected the **word-level labels** of PAN-X with the **token-level input** of XLM-R. The same rule is used everywhere: label the first sub-token of each word and mark all other positions with `-100`.

| Step | Result |
|---|---|
| Tokenization of the first German example | 12 words became 25 tokens, including `<s>` and `</s>` |
| Word ids | Tokens of one word share the same index, special tokens have `None` |
| Label alignment | One real label per word, `-100` for special tokens and continuation sub-tokens |
| `tokenized_and_align_labels` | Applies the rule to a batch and adds `labels` |
| `encode_panx_dataset` | Encodes train, validation and test splits (12,580 / 6,290 / 6,290 for German) |

### What comes next

The next notebook, `create_model.ipynb`, builds the XLM-R model for token classification. It will produce one prediction for each token, and the `-100` positions we created here will be ignored by the loss.

## **Resources & References**

**Official Documentation**

- [Hugging Face Transformers: Token classification guide](https://huggingface.co/docs/transformers/tasks/token_classification)
- [Hugging Face Transformers: Tokenization algorithms](https://huggingface.co/docs/transformers/tokenizer_summary)
- [Hugging Face Datasets: Process (`map`, `shuffle`, `select`)](https://huggingface.co/docs/datasets/process)
- [PyTorch: `CrossEntropyLoss` (see `ignore_index`)](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

**Key Papers / Research**

- Kudo, T., Richardson, J. (2018). [SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing](https://arxiv.org/abs/1808.06226).
- Conneau, A. et al. (2019). [Unsupervised Cross-lingual Representation Learning at Scale](https://arxiv.org/abs/1911.02116).

**Additional Reading**

- Tunstall, L., von Werra, L., Wolf, T. (2022). [Natural Language Processing with Transformers, Revised Edition](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/). O'Reilly Media. Chapter 4: Multilingual Named Entity Recognition.